# Tema 10 — Preprocesamiento y aumentación de imágenes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-05/Tema-10/Tema_10.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

> **Requisitos:** `opencv-python`, `numpy`, `matplotlib`, `albumentations`. Coloca la imagen `tigre.png` en el mismo directorio del notebook.


## 0. Setup (Colab / local)

Esta celda se asegura de que la imagen `tigre.png` esté disponible en el directorio de trabajo. En **Google Colab** la descarga desde GitHub; en **local** la usa si ya existe. También instala `opencv-python` y `albumentations` si hace falta.


In [ ]:
# --- Setup: descarga la imagen si no existe (funciona en Colab y en local) ---
import os
import sys
import subprocess
import urllib.request

# Instalar dependencias si no están disponibles (Colab trae cv2 pero no albumentations)
def _ensure(paquete, import_name=None):
    nombre = import_name or paquete
    try:
        __import__(nombre)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_ensure("opencv-python", "cv2")
_ensure("albumentations")

REPO_RAW = "https://raw.githubusercontent.com/ecamposv/nlp-vision/main/semana-05/Tema-10"
IMG = "tigre.png"

if not os.path.exists(IMG):
    urllib.request.urlretrieve(f"{REPO_RAW}/{IMG}", IMG)
    print(f"[Descargado] {IMG} ({os.path.getsize(IMG)} bytes)")
else:
    print(f"[OK] {IMG} ya existe ({os.path.getsize(IMG)} bytes)")


## 1. Carga, redimensionamiento y normalización inicial

La celda muestra el flujo mínimo para preparar una imagen para un modelo de visión:

1. **Carga** la imagen con `cv2.imread("tigre.png")` (formato BGR).
2. **Elige la interpolación** según el caso: `cv2.INTER_AREA` para reducir (mejor calidad al hacer downsample) y `cv2.INTER_CUBIC` para ampliar.
3. **Redimensiona** a `256×256` con `cv2.resize`.
4. **Normaliza** a `float32` en el rango `[0, 1]` dividiendo entre `255.0` — formato esperado por la mayoría de redes neuronales.
5. Imprime `shape`, `dtype` y los valores mínimo/máximo para verificar la normalización.
6. Guarda la imagen redimensionada en disco con `cv2.imwrite`.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Lee la imagen (OpenCV usa BGR por defecto)
img = cv2.imread("tigre.png")
if img is None:
    raise FileNotFoundError("No se encontró 'tigre.png'. Ejecuta primero la celda de Setup.")
# Alias usado por celdas posteriores
img_bgr = img
# Redimensiona a 256x256 (elige interpolación según si reduces o amplías)
h, w = img.shape[:2]
inter = cv2.INTER_AREA if max(h, w) > 256 else cv2.INTER_CUBIC
resized = cv2.resize(img, (256, 256), interpolation=inter)
# Normaliza a [0,1] en float32
norm_01 = resized.astype(np.float32) / 255.0
print("Shape:", norm_01.shape, "dtype:", norm_01.dtype, "min/max:", norm_01.min(), norm_01.max())
# (Opcional) Guardar la versión redimensionada (sin normalización visual)
cv2.imwrite("salida_redimensionada.jpg", resized)

## 2. Normalización por escalamiento y estadísticas

La celda implementa el método más simple de normalización ($\tilde{x} = x / 255$) y calcula las estadísticas que típicamente se usan para una normalización más avanzada (estandarización por media/std).

**Qué hace paso a paso:**

1. Escala todos los píxeles a `[0, 1]` en `float32`.
2. Si la imagen es a color, calcula la **media y desviación estándar por canal** (B, G, R) usando `axis=(0,1)` para promediar sobre todos los píxeles.
3. Calcula también estadísticas **globales** (un único valor para toda la imagen).
4. Convierte BGR → RGB invirtiendo el último eje (`[:, :, ::-1]`) para visualizar correctamente.
5. Muestra la imagen normalizada con `plt.imshow` y la **guarda** convirtiendo de nuevo a `uint8` (`*255` + `clip` + `round`).

Estas estadísticas son útiles para datasets propios cuando no se quiere usar la media/std de ImageNet.


In [ ]:
# --- 2) Normalización por escalamiento a [0,1]: x̃ = x/255 ---
x01_bgr = img.astype(np.float32) / 255.0  # HxWxC o HxW, en [0,1]
# --- 3) Cálculo de estadísticas: media y desviación estándar ---
if x01_bgr.ndim == 3 and x01_bgr.shape[2] == 3:
    # Por canal (B, G, R) en [0,1]
    mean_c = x01_bgr.mean(axis=(0, 1))
    std_c  = x01_bgr.std(axis=(0, 1))
    print("== Estadísticas por canal (B, G, R) en [0,1] ==")
    print("Media : ", mean_c)
    print("Std   : ", std_c)
    # Global (todos los píxeles y canales)
    mean_g = float(x01_bgr.mean())
    std_g  = float(x01_bgr.std())
    print("\n== Estadísticas globales (todos los píxeles) en [0,1] ==")
    print(f"Media: {mean_g:.6f} | Std: {std_g:.6f}")
    # Para visualizar correctamente: BGR -> RGB
    x01_vis = x01_bgr[:, :, ::-1]  # RGB en [0,1]
else:
    # Imagen en escala de grises
    mean_g = float(x01_bgr.mean())
    std_g  = float(x01_bgr.std())
    print("== Estadísticas (grises) en [0,1] ==")
    print(f"Media: {mean_g:.6f} | Std: {std_g:.6f}")
    x01_vis = x01_bgr  # 2D en [0,1] para mostrar en escala de grises
# --- 4) Mostrar la imagen normalizada ---
plt.figure()
if x01_vis.ndim == 2:
    plt.imshow(x01_vis, vmin=0, vmax=1, cmap="gray")
else:
    plt.imshow(x01_vis)  # RGB en [0,1]
plt.title("Imagen normalizada a [0,1] (x̃ = x/255)")
plt.axis("off")
plt.show()
# --- 5) Guardar la imagen normalizada como 8-bit para visualización/compartir ---
out_8u = (np.clip(x01_bgr, 0.0, 1.0) * 255.0).round().astype(np.uint8)
cv2.imwrite("salida_normalizada_01.jpg", out_8u)
print("Guardada: salida_normalizada_01.jpg")

## 3. Transformaciones geométricas y de espacio de color

La celda muestra tres formas distintas de **adaptar el tamaño** de una imagen y dos **conversiones de espacio de color**, todas comparadas en un solo gráfico.

**Bloques:**

1. **Center crop** a 224×224 — si la imagen es más pequeña que el crop, primero se escala con `cv2.resize` usando el factor `max(ch/h, cw/w)` para asegurar cobertura; después se recorta el centro.
2. **Letterbox** (padding manteniendo aspecto) — calcula `scale = min(tw/w, th/h)` para que la imagen entre completa, la redimensiona y rellena los bordes con gris `(114, 114, 114)` usando `np.full`. Este es el método estándar en YOLO y similares.
3. **Conversiones de color** — `BGR → RGB` (visualización) y `RGB → GRAY` (simplificación a 1 canal).
4. **Visualización comparativa** — `plt.subplot(2, 3, ...)` muestra original, center crop, letterbox, RGB y grises en una sola figura que se guarda como `salida_comparativa_procesos.png`.

**Importante:** esta celda asume que existe la variable `img_bgr` (definida en celdas anteriores o renombrada desde `img`).


In [ ]:
h, w = img_bgr.shape[:2]
# ==============================================================
# 1) Recorte centrado (center crop) a un tamaño objetivo (ch, cw)
#    - Si la imagen es más pequeña, primero se escala para cubrir.
# ==============================================================
ch, cw = 224, 224  # tamaño de recorte deseado
if h < ch or w < cw:
    # Escalar para asegurar que la imagen cubra el crop
    scale = max(ch / h, cw / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    inter = cv2.INTER_CUBIC if scale > 1.0 else cv2.INTER_AREA
    img_bgr_scaled = cv2.resize(img_bgr, (nw, nh), interpolation=inter)
else:
    img_bgr_scaled = img_bgr.copy()
hs, ws = img_bgr_scaled.shape[:2]
y0 = (hs - ch) // 2
x0 = (ws - cw) // 2
center_crop_bgr = img_bgr_scaled[y0:y0+ch, x0:x0+cw]
print(f"Center crop BGR: {center_crop_bgr.shape}")
# ==============================================================================
# 2) Padding para mantener la relación de aspecto (letterbox) a tamaño (th, tw)
#    - Redimensiona manteniendo aspecto y rellena los bordes para llegar al tamaño.
# ==============================================================================
th, tw = 224, 224
scale = min(tw / w, th / h)
nw, nh = int(round(w * scale)), int(round(h * scale))
inter = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
resized_bgr = cv2.resize(img_bgr, (nw, nh), interpolation=inter)

pad_color = (114, 114, 114)  # gris suave típico
letterbox_bgr = np.full((th, tw, 3), pad_color, dtype=resized_bgr.dtype)
y0 = (th - nh) // 2
x0 = (tw - nw) // 2
letterbox_bgr[y0:y0+nh, x0:x0+nw] = resized_bgr
print(f"Letterbox BGR: {letterbox_bgr.shape}, scale={scale:.4f}, pad_color={pad_color}")
# ======================================================
# 3) Conversiones de espacio de color
#    - BGR -> RGB (común para frameworks/visualización)
#    - RGB -> GRAY (escala de grises)
# ======================================================
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
print(f"RGB: {img_rgb.shape}, GRAY: {img_gray.shape}")
# ============================================
# 4) Visualización comparativa con matplotlib
#     (matplotlib muestra en RGB, no en BGR)
# ============================================
plt.figure(figsize=(14, 6))
plt.subplot(2, 3, 1)
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original (BGR→RGB)")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(cv2.cvtColor(center_crop_bgr, cv2.COLOR_BGR2RGB))
plt.title("Recorte centrado")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(cv2.cvtColor(letterbox_bgr, cv2.COLOR_BGR2RGB))
plt.title("Letterbox (padding AR)")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(img_rgb)
plt.title("Conversión BGR→RGB")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(img_gray, cmap="gray", vmin=0, vmax=255)
plt.title("RGB→Grises")
plt.axis("off")

plt.tight_layout()
plt.savefig("salida_comparativa_procesos.png")
plt.show()

## 4. Aumentación de datos (transformaciones espaciales)

La celda aplica **cinco categorías** de aumentos geométricos típicos en *data augmentation* para entrenar modelos de visión. Cada transformación se hace **a mano con OpenCV/NumPy** para mostrar cómo funcionan por dentro.

1. **Rotación** — `cv2.getRotationMatrix2D(center, 15°, 1.0)` + `cv2.warpAffine` con `borderMode=BORDER_REFLECT_101` para evitar bordes negros.
2. **Traslación** — matriz afín `[[1,0,tx],[0,1,ty]]` con `tx=40, ty=-20`.
3. **Flips** — `cv2.flip(img, 1)` horizontal y `cv2.flip(img, 0)` vertical.
4. **Zoom in** (×1.25) — escala y recorta al centro para mantener el tamaño original.
5. **Zoom out** (×0.75) — escala hacia abajo y rellena con padding gris para volver al tamaño original.
6. **Random crop** — usa `np.random.seed(42)` (reproducible), toma un recorte del 80 % en posición aleatoria y lo reescala al tamaño original.

Finalmente muestra las 8 versiones (original + 7 aumentos) en una cuadrícula `2×4` y la guarda como `salida_augmentations.png`.


In [ ]:
# ============================================
# 1) Giros (rotaciones) alrededor del centro
# ============================================
angle = 15.0  # grados
center = (w / 2.0, h / 2.0)
M_rot = cv2.getRotationMatrix2D(center, angle, 1.0)
rotated = cv2.warpAffine(
    img, M_rot, (w, h),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_REFLECT_101
)
# ============================================
# 2) Traslaciones (desplazamientos x,y)
# ============================================
tx, ty = 40, -20  # píxeles (+x derecha, +y abajo)
M_trans = np.float32([[1, 0, tx], [0, 1, ty]])
translated = cv2.warpAffine(
    img, M_trans, (w, h),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_REFLECT_101
)
# ============================================
# 3) Reflejos (flips)
#    1 = horizontal, 0 = vertical, -1 = ambos
# ============================================
flip_h = cv2.flip(img, 1)
flip_v = cv2.flip(img, 0)
# ============================================
# 4) Zoom
#    - Zoom in (>1): escalar y recortar al centro
#    - Zoom out (<1): escalar y rellenar (letterbox)
# ============================================
zoom_in_factor = 1.25
zw_in, zh_in = int(round(w * zoom_in_factor)), int(round(h * zoom_in_factor))
img_zoom_in = cv2.resize(
    img, (zw_in, zh_in),
    interpolation=cv2.INTER_CUBIC
)
# recorte centrado de vuelta a (h, w)
y0 = (zh_in - h) // 2
x0 = (zw_in - w) // 2
zoom_in = img_zoom_in[y0:y0+h, x0:x0+w]
# ---- Zoom out (alejar)
zoom_out_factor = 0.75
zw_out, zh_out = int(round(w * zoom_out_factor)), int(round(h * zoom_out_factor))
img_zoom_out_small = cv2.resize(
    img, (zw_out, zh_out),
    interpolation=cv2.INTER_AREA
)
# canvas con padding para volver a (h, w)
pad_color = (114, 114, 114)
zoom_out = np.full((h, w, 3), pad_color, dtype=img.dtype)
y0 = (h - zh_out) // 2
x0 = (w - zw_out) // 2
zoom_out[y0:y0+zh_out, x0:x0+zw_out] = img_zoom_out_small
# ============================================
# 5) Recortes aleatorios (random crop)
#    - Tomamos 80% del tamaño original, posición aleatoria
#    - Luego reescalamos al tamaño original para visualizar/usar
# ============================================
np.random.seed(42)  # reproducible
crop_ratio = 0.8
ch = max(1, int(round(h * crop_ratio)))
cw = max(1, int(round(w * crop_ratio)))
y0 = np.random.randint(0, h - ch + 1) if h - ch >= 0 else 0
x0 = np.random.randint(0, w - cw + 1) if w - cw >= 0 else 0
crop = img[y0:y0+ch, x0:x0+cw]
random_crop = cv2.resize(crop, (w, h), interpolation=cv2.INTER_LINEAR)
# ============================================
# Visualización (matplotlib usa RGB)
# ============================================
def bgr2rgb(a):
    return a[..., ::-1]
titles = [
    "Original", "Giro +15°", "Traslación (40,-20)", "Flip Horizontal",
    "Flip Vertical", "Zoom In (1.25x)", "Zoom Out (0.75x)", "Recorte aleatorio"
]
images = [
    img, rotated, translated, flip_h,
    flip_v, zoom_in, zoom_out, random_crop
]
plt.figure(figsize=(14, 8))
for i, (im, t) in enumerate(zip(images, titles), 1):
    plt.subplot(2, 4, i)
    plt.imshow(bgr2rgb(im))
    plt.title(t)
    plt.axis("off")
plt.tight_layout()
plt.savefig("salida_augmentations.png")
plt.show()

## 5. Transformaciones fotométricas

A diferencia de las geométricas, estas alteran los **valores de los píxeles** (color e iluminación) sin cambiar la posición de los objetos. Son muy útiles para entrenar modelos más robustos a cambios de iluminación.

1. **Brillo** — sube/baja el **canal V** en HSV multiplicándolo por `b_factor=1.20` (+20 %) y haciendo `np.clip(..., 0, 255)`.
2. **Contraste** — fórmula clásica `out = α·(x − 127.5) + 127.5` con `α=1.30` (>1 aumenta, <1 reduce).
3. **Saturación** — escala el **canal S** en HSV por `s_factor=1.40`.
4. **Matiz (hue)** — desplaza el **canal H** en 15° estándar (≈ `delta_h = 7` en la escala 0–179 de OpenCV) y aplica módulo 180 para que sea cíclico. Se usa `int16` temporalmente para evitar overflow.
5. **Ruido gaussiano** — añade ruido `N(0, σ=0.03)` sobre la imagen normalizada en `[0,1]` y vuelve a `uint8`.

La visualización en `2×3` compara las 5 transformaciones contra la original y guarda el resultado en `salida_augmentations.png`.


In [ ]:
# ===============================
# 1) Cambio de brillo (HSV: canal V)
#    factor > 1 = más brillo, < 1 = menos
# ===============================
b_factor = 1.20  # +20% brillo
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
hsv[..., 2] = np.clip(hsv[..., 2] * b_factor, 0, 255)  # V
bright_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
# ===============================
# 2) Cambio de contraste (BGR)
#    out = alpha*(x - 127.5) + 127.5
# ===============================
alpha = 1.30  # >1 aumenta contraste, <1 reduce
img32 = img_bgr.astype(np.float32)
contrast_bgr = np.clip(alpha * (img32 - 127.5) + 127.5, 0, 255).astype(np.uint8)
# ===============================
# 3) Cambio de saturación (HSV: canal S)
#    factor > 1 = más saturación
# ===============================
s_factor = 1.40
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
hsv[..., 1] = np.clip(hsv[..., 1] * s_factor, 0, 255)  # S
saturated_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
# ===============================
# 4) Cambio de matiz / hue (HSV: canal H)
#    OpenCV: H ∈ [0,179] ~ grados/2
# ===============================
delta_deg = 15               # rotación de matiz en grados estándar
delta_h = int(round(delta_deg / 2))  # a escala OpenCV
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
h_channel = hsv[..., 0].astype(np.int16)  # usar tipo ancho para evitar overflow
h_channel = (h_channel + delta_h) % 180
hsv[..., 0] = h_channel.astype(np.uint8)
hue_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
# ===============================
# 5) Adición de ruido gaussiano (BGR, en [0,1])
# ===============================
np.random.seed(0)  # reproducible
x01 = img_bgr.astype(np.float32) / 255.0
sigma = 0.03  # desviación estándar del ruido
noise = np.random.normal(0.0, sigma, size=x01.shape).astype(np.float32)
noisy01 = np.clip(x01 + noise, 0.0, 1.0)
noisy_bgr = (noisy01 * 255.0).round().astype(np.uint8)
# ===============================
# Visualización (matplotlib usa RGB)
# ===============================
def bgr2rgb(a):
    return a[..., ::-1]
titles = [
    "Original",
    "Brillo +20% (HSV·V)",
    "Contraste ×1.3",
    "Saturación ×1.4 (HSV·S)",
    "Matiz +15° (HSV·H)",
    "Ruido Gaussiano (σ=0.03)"
]
images = [
    img_bgr,
    bright_bgr,
    contrast_bgr,
    saturated_bgr,
    hue_bgr,
    noisy_bgr
]
plt.figure(figsize=(14, 7))
for i, (im, t) in enumerate(zip(images, titles), 1):
    plt.subplot(2, 3, i)
    plt.imshow(bgr2rgb(im))
    plt.title(t)
    plt.axis("off")
plt.tight_layout()
plt.savefig("salida_augmentations.png")
plt.show()

## 6. Generación de ruido y filtros de eliminación de ruido

La celda **genera dos tipos de ruido sintético** y luego aplica **tres filtros** de denoising para comparar cuál funciona mejor en cada caso.

**Tipos de ruido:**

- **Ruido gaussiano** (aditivo): `N(0, σ=0.06)` sumado a la imagen en `[0,1]`. Simula ruido de sensor en baja luz.
- **Ruido sal y pimienta** (impulsivo): píxeles aleatorios puestos a 1 (sal) o 0 (pimienta) con probabilidad `p=0.03`. Simula píxeles defectuosos o errores de transmisión.

**Filtros aplicados a cada imagen ruidosa:**

- **`cv2.GaussianBlur(k=5, σ=1.2)`** — suavizado lineal, bueno contra ruido gaussiano pero **borra bordes**.
- **`cv2.medianBlur(k=5)`** — filtro **no lineal** que toma la mediana; es el más eficaz contra **sal y pimienta** porque ignora valores extremos.
- **`cv2.bilateralFilter(d=9, σ_color=75, σ_space=75)`** — suaviza **preservando bordes** considerando proximidad espacial y similitud de color.

> **Regla práctica:** para sal y pimienta → mediana; para ruido gaussiano suave → bilateral si quieres conservar bordes.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
# ===============================
# 0) Cargar imagen (BGR)
# ===============================
IMG_PATH = "tigre.png"  # cambia por tu imagen
img_bgr = cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)
img_bgr = cv2.resize(img_bgr, (256, 256), interpolation=cv2.INTER_AREA)
h, w = img_bgr.shape[:2]
# ===============================
# 1) Crear imágenes con ruido
#    a) Ruido gaussiano (aditivo)
#    b) Ruido sal y pimienta (impulsivo)
# ===============================
np.random.seed(0)  # reproducible
# a) Ruido gaussiano en [0,1]
x01 = img_bgr.astype(np.float32) / 255.0
sigma = 0.06  # desviación estándar del ruido (ajústala)
noise = np.random.normal(0.0, sigma, size=x01.shape).astype(np.float32)
noisy_gauss01 = np.clip(x01 + noise, 0.0, 1.0)
noisy_gauss_bgr = (noisy_gauss01 * 255.0).round().astype(np.uint8)
# b) Ruido sal y pimienta
p = 0.03  # proporción de píxeles alterados (ajústala)
sp01 = x01.copy()
mask = np.random.rand(h, w)
salt = mask < (p / 2.0)
pepper = mask > (1.0 - p / 2.0)
sp01[salt] = 1.0
sp01[pepper] = 0.0
noisy_sp_bgr = (sp01 * 255.0).round().astype(np.uint8)
# ===============================
# 2) Filtros de eliminación de ruido
#    - Gaussiano (cv2.GaussianBlur): suavizado suave
#    - Mediana (cv2.medianBlur): muy eficaz contra sal-pimienta
#    - Bilateral (cv2.bilateralFilter): suaviza preservando bordes
# ===============================
# --- Sobre imagen con ruido gaussiano ---
gauss_on_gauss = cv2.GaussianBlur(noisy_gauss_bgr, ksize=(5, 5), sigmaX=1.2, borderType=cv2.BORDER_REFLECT_101)
median_on_gauss = cv2.medianBlur(noisy_gauss_bgr, ksize=5)
bilat_on_gauss  = cv2.bilateralFilter(noisy_gauss_bgr, d=9, sigmaColor=75, sigmaSpace=75)
# --- Sobre imagen con ruido sal y pimienta ---
gauss_on_sp = cv2.GaussianBlur(noisy_sp_bgr, ksize=(5, 5), sigmaX=1.2, borderType=cv2.BORDER_REFLECT_101)
median_on_sp = cv2.medianBlur(noisy_sp_bgr, ksize=5)
bilat_on_sp  = cv2.bilateralFilter(noisy_sp_bgr, d=9, sigmaColor=75, sigmaSpace=75)

## 7. Pipeline de aumentación con Albumentations

En la práctica no se programan los aumentos a mano: se usa una librería como **Albumentations**, que aplica cada transformación con una **probabilidad `p`** y permite encadenarlas en un solo `A.Compose`. Cada vez que se llama al pipeline se genera una variante aleatoria (simulando una época de entrenamiento).

**Transformaciones del pipeline:**

| Transformación | Parámetros clave | Qué hace |
|---|---|---|
| `RandomResizedCrop` | `size=(224,224)`, `scale=(0.8, 1.0)` | Recorte aleatorio + resize al tamaño objetivo |
| `HorizontalFlip` | `p=0.5` | Espejo horizontal con 50 % de probabilidad |
| `VerticalFlip` | `p=0.1` | Espejo vertical con 10 % |
| `ShiftScaleRotate` | shift ±6 %, scale ±15 %, rot ±25° | Traslación + escala + rotación combinadas |
| `RandomBrightnessContrast` | ±0.2 | Cambios de brillo y contraste |
| `HueSaturationValue` | hue ±10, sat ±20, val ±10 | Variación de color en HSV |
| `GaussNoise` | `var_limit=(5, 25)` | Ruido gaussiano leve |

El pipeline recibe la imagen en **RGB** (por eso se convierte primero con `cv2.cvtColor`), devuelve un dict y se accede al resultado con `out = transform(image=img_rgb)["image"]`.

> ⚠️ En **Albumentations 2.x** algunas APIs cambiaron: `ShiftScaleRotate` ahora es `A.Affine` y `GaussNoise(var_limit=...)` se reemplaza por `std_range=(...)`. Si tu versión es 2.x, ajusta el código.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
# ===============================
# 1) Definir pipeline de aumentos
#    - Cada transformación tiene probabilidad p
#    - Ajustar parámetros
# ===============================
IMG_PATH = "tigre.png"  # cambia por tu imagen
# --- Cargar imagen y pasar a RGB ---
img_bgr = cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
# --- Definir tamaño objetivo y pipeline (Albumentations v2.x) ---
H, W = 224, 224
transform = A.Compose([
    A.RandomResizedCrop(size=(H, W), scale=(0.8, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.15, rotate_limit=25,
                       border_mode=cv2.BORDER_REFLECT_101, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),
], p=1.0)
# --- Aplicar el pipeline (simula una época) ---
out = transform(image=img_rgb)["image"]
# --- Mostrar y guardar ---
plt.imshow(out)
plt.title("Augment (Albumentations)")
plt.axis("off")
plt.savefig("Augmented_Albumetations_01.png")
plt.show()
print("Guardada: aug_sample.jpg")
